# Chapter 3: Linear Models for Regression

In this chapter, I will explain **Linear Models for Regression** using a simple and practical example: **predicting the price of a used car from its mileage and other features**.

The goal is to understand the idea first, then connect it to code step by step.

## What you will learn

By the end of this notebook, you should be able to explain and implement:

1. What regression means
2. What a linear regression model is
3. How a model makes predictions
4. What model parameters are
5. What a cost function is
6. How least squares finds the best line
7. How gradient descent learns model parameters
8. How to evaluate regression models
9. How polynomial features are still linear models
10. How regularization helps reduce overfitting

---


## 1. What Is Regression?

In Machine Learning, **regression** means predicting a **continuous numeric value**.

For example:

| Problem | Input | Output |
|---|---|---|
| Used car price prediction | Mileage, year, brand, distance travelled | Price |
| House price prediction | Area, location, number of rooms | Price |
| Salary prediction | Experience, education, role | Salary |
| Temperature forecasting | Date, humidity, pressure | Temperature |

In this chapter, I will use a simple used-car example.

Suppose we have data about cars:

| Mileage in km/l | Car Price in lakhs |
|---:|---:|
| 10 | 4.0 |
| 12 | 4.8 |
| 14 | 5.4 |
| 16 | 6.2 |
| 18 | 7.1 |

Here, mileage is the input feature `x`, and car price is the output value `y`.

### In plain English

Regression is used when the answer is a number. The model studies old examples and predicts a new numeric value.

## 2. The Simple Idea Behind Linear Regression

Linear regression assumes that the output can be approximately represented by a straight line.

For one input feature, the equation is:

$$
\hat{y} = w_0 + w_1x
$$

Where:

| Symbol | Meaning |
|---|---|
| $x$ | Input feature, such as mileage |
| $\hat{y}$ | Predicted output, such as predicted car price |
| $w_0$ | Intercept or bias |
| $w_1$ | Slope or weight |

The model learns the values of $w_0$ and $w_1$ from data.

### Meaning of slope and intercept

If the model learns this equation:

$$
\hat{y} = 1.5 + 0.30x
$$

Then:

- `1.5` is the base price component.
- `0.30` means that for every 1 km/l increase in mileage, the predicted price increases by 0.30 lakhs.

### In plain English

Linear regression tries to draw the best straight line through the data points.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Make plots slightly larger
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

## 3. Creating a Simple Dataset

I will start with a very small dataset.

This is not a real production dataset. It is intentionally small so that the idea is easy to understand.

In [ ]:
# Simple used-car dataset
# x = mileage in km/l
# y = car price in lakhs

data = {
    "Mileage_kmpl": [10, 12, 14, 16, 18, 20, 22],
    "Price_lakhs":  [4.0, 4.8, 5.4, 6.2, 7.1, 7.8, 8.7]
}

df = pd.DataFrame(data)
df

## 4. Visualizing the Data

Before fitting any model, I always like to plot the data.

A plot helps us answer an important question:

> Does the relationship look roughly linear?

If the points roughly follow a straight-line trend, linear regression can be a reasonable starting point.

In [ ]:
plt.scatter(df["Mileage_kmpl"], df["Price_lakhs"])
plt.xlabel("Mileage in km/l")
plt.ylabel("Car price in lakhs")
plt.title("Mileage vs Car Price")
plt.grid(True)
plt.show()

## 5. Prediction Using a Line

Let us manually choose a line:

$$
\hat{y} = 0.5 + 0.38x
$$

This means:

- Base price component = `0.5` lakhs
- Price increases by `0.38` lakhs for every 1 km/l increase in mileage

Now I will use this line to make predictions.

In [ ]:
w0 = 0.5
w1 = 0.38

x = df["Mileage_kmpl"].values
y_actual = df["Price_lakhs"].values

y_pred_manual = w0 + w1 * x

df_manual = df.copy()
df_manual["Predicted_Price"] = y_pred_manual
df_manual["Error"] = df_manual["Predicted_Price"] - df_manual["Price_lakhs"]
df_manual

In [ ]:
plt.scatter(x, y_actual, label="Actual data")
plt.plot(x, y_pred_manual, label="Manual line")
plt.xlabel("Mileage in km/l")
plt.ylabel("Car price in lakhs")
plt.title("Manual Linear Model")
plt.legend()
plt.grid(True)
plt.show()

## 6. What Is the Error?

For every data point, the model makes a prediction.

The difference between the actual value and predicted value is called the **residual** or **error**.

$$
\text{error} = \hat{y} - y
$$

Where:

- $\hat{y}$ is predicted value
- $y$ is actual value

If the error is small, the prediction is close to the actual value.

If the error is large, the prediction is poor.

### In plain English

Error tells us how far the model prediction is from the correct answer.

In [ ]:
df_manual[["Mileage_kmpl", "Price_lakhs", "Predicted_Price", "Error"]]

## 7. Cost Function: Measuring Overall Mistake

A single error tells us the mistake for one row.

But a model must work well on the whole dataset. So we need one number that summarizes the total mistake.

A common cost function for linear regression is **Mean Squared Error**.

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

Why square the error?

1. Negative and positive errors do not cancel each other.
2. Larger mistakes are punished more.
3. The function becomes convenient for optimization.

### In plain English

The cost function is the model's overall penalty. Lower cost means better fit.

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

manual_mse = mse(y_actual, y_pred_manual)
manual_mse

## 8. Learning the Best Line Using Least Squares

Instead of manually guessing $w_0$ and $w_1$, linear regression learns them from data.

The goal is:

> Find the line that minimizes the sum of squared errors.

This is called the **least squares** approach.

For simple linear regression with one feature, the best slope and intercept can be calculated directly:

$$
w_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}
$$

$$
w_0 = \bar{y} - w_1\bar{x}
$$

Here:

- $\bar{x}$ is the mean of x values
- $\bar{y}$ is the mean of y values

### In plain English

Least squares finds the straight line that keeps the total squared prediction error as small as possible.

In [ ]:
x_mean = np.mean(x)
y_mean = np.mean(y_actual)

w1_best = np.sum((x - x_mean) * (y_actual - y_mean)) / np.sum((x - x_mean) ** 2)
w0_best = y_mean - w1_best * x_mean

print("Best intercept w0:", round(w0_best, 4))
print("Best slope w1:", round(w1_best, 4))

In [ ]:
y_pred_best = w0_best + w1_best * x

print("Manual guessed MSE:", round(manual_mse, 4))
print("Least squares MSE:", round(mse(y_actual, y_pred_best), 4))

In [ ]:
plt.scatter(x, y_actual, label="Actual data")
plt.plot(x, y_pred_manual, linestyle="--", label="Manual line")
plt.plot(x, y_pred_best, label="Least squares best line")
plt.xlabel("Mileage in km/l")
plt.ylabel("Car price in lakhs")
plt.title("Manual Line vs Least Squares Line")
plt.legend()
plt.grid(True)
plt.show()

## 9. Fitting Linear Regression Using Scikit-Learn

In real projects, we usually use libraries such as `scikit-learn`.

The library internally learns the best parameters.

In [ ]:
X = df[["Mileage_kmpl"]]   # Input must be 2D for sklearn
y = df["Price_lakhs"]      # Output

model = LinearRegression()
model.fit(X, y)

print("Intercept:", round(model.intercept_, 4))
print("Slope:", round(model.coef_[0], 4))

In [ ]:
# Predict price for a new car with mileage = 19 km/l
new_car = pd.DataFrame({"Mileage_kmpl": [19]})
predicted_price = model.predict(new_car)[0]

print("Predicted price for mileage 19 km/l:", round(predicted_price, 2), "lakhs")

## 10. Model Parameters

In linear regression, the numbers learned by the model are called **parameters**.

For a simple model:

$$
\hat{y} = w_0 + w_1x
$$

The parameters are:

- $w_0$: intercept
- $w_1$: slope

For multiple input features:

$$
\hat{y} = w_0 + w_1x_1 + w_2x_2 + w_3x_3 + ... + w_dx_d
$$

The model learns one weight for each feature.

### Example

For car price prediction:

$$
\text{Price} = w_0 + w_1(\text{Mileage}) + w_2(\text{Age}) + w_3(\text{Distance})
$$

### In plain English

Parameters are the numbers the model adjusts to make better predictions.

## 11. Multiple Linear Regression

Simple linear regression uses one input feature.

Multiple linear regression uses more than one feature.

For used-car price prediction, mileage alone may not be enough. Price may also depend on:

- Car age
- Distance travelled
- Engine capacity
- Brand
- Condition

Here I will create a slightly richer example using three numeric features:

1. Mileage
2. Age of car
3. Distance travelled

In [ ]:
multi_data = {
    "Mileage_kmpl": [10, 12, 14, 16, 18, 20, 22, 15, 13, 19],
    "Age_years":    [9, 8, 7, 6, 5, 4, 3, 6, 7, 4],
    "Distance_km":  [90000, 80000, 70000, 62000, 52000, 40000, 30000, 65000, 75000, 42000],
    "Price_lakhs":  [3.5, 4.2, 4.9, 5.8, 6.6, 7.6, 8.5, 5.4, 4.6, 7.2]
}

df_multi = pd.DataFrame(multi_data)
df_multi

In [ ]:
X_multi = df_multi[["Mileage_kmpl", "Age_years", "Distance_km"]]
y_multi = df_multi["Price_lakhs"]

multi_model = LinearRegression()
multi_model.fit(X_multi, y_multi)

params = pd.DataFrame({
    "Feature": ["Intercept"] + list(X_multi.columns),
    "Parameter": [multi_model.intercept_] + list(multi_model.coef_)
})
params

### How to interpret the coefficients

Each coefficient tells how the prediction changes when that feature increases by one unit, assuming other features are fixed.

For example:

- A positive mileage coefficient means higher mileage is associated with higher predicted price.
- A negative age coefficient means older cars are associated with lower predicted price.
- A negative distance coefficient means cars that travelled more distance are associated with lower predicted price.

### Important warning

Coefficient interpretation is meaningful only when:

1. The data is reasonably clean.
2. Features are not highly misleading.
3. The model assumptions are acceptable.
4. Units of measurement are understood.

### In plain English

Multiple regression predicts using many clues instead of only one clue.

## 12. Gradient Descent: Learning Parameters Iteratively

The least-squares formula gives a direct solution.

But for large datasets and complex models, we often use an iterative optimization method called **gradient descent**.

The idea is simple:

1. Start with random or zero values for the weights.
2. Make predictions.
3. Calculate error.
4. Move the weights in the direction that reduces error.
5. Repeat until the error becomes small.

The update rule is:

$$
w_j := w_j - \alpha \frac{\partial J}{\partial w_j}
$$

Where:

| Symbol | Meaning |
|---|---|
| $w_j$ | model parameter |
| $J$ | cost function |
| $\alpha$ | learning rate |
| $\frac{\partial J}{\partial w_j}$ | gradient or direction of steepest increase |

Because we subtract the gradient, we move toward lower cost.

### In plain English

Gradient descent is like walking downhill on an error surface until we reach a low-error point.

In [ ]:
# Implement gradient descent for simple linear regression

X_gd = df["Mileage_kmpl"].values.astype(float)
y_gd = df["Price_lakhs"].values.astype(float)

# Feature scaling helps gradient descent converge smoothly
X_mean = X_gd.mean()
X_std = X_gd.std()
X_scaled = (X_gd - X_mean) / X_std

# Initialize parameters
w0 = 0.0
w1 = 0.0
learning_rate = 0.05
epochs = 200
n = len(X_scaled)

cost_history = []

for epoch in range(epochs):
    y_pred = w0 + w1 * X_scaled
    error = y_pred - y_gd
    
    cost = np.mean(error ** 2)
    cost_history.append(cost)
    
    # Gradients of MSE
    dw0 = (2 / n) * np.sum(error)
    dw1 = (2 / n) * np.sum(error * X_scaled)
    
    # Parameter update
    w0 = w0 - learning_rate * dw0
    w1 = w1 - learning_rate * dw1

print("Learned w0:", round(w0, 4))
print("Learned w1:", round(w1, 4))
print("Final cost:", round(cost_history[-1], 6))

In [ ]:
plt.plot(cost_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Cost")
plt.title("Gradient Descent: Cost Decreases Over Time")
plt.grid(True)
plt.show()

## 13. Learning Rate

The learning rate controls the size of each step in gradient descent.

| Learning Rate | Effect |
|---|---|
| Too small | Training is very slow |
| Good value | Cost decreases smoothly |
| Too large | Training may jump around or diverge |

A learning rate is a **hyperparameter**. It is not learned automatically in basic gradient descent. We choose it and tune it.

### In plain English

Learning rate decides whether the model takes baby steps, useful steps, or dangerous jumps.

In [ ]:
def run_gradient_descent(alpha, epochs=80):
    w0, w1 = 0.0, 0.0
    history = []
    n = len(X_scaled)
    
    for _ in range(epochs):
        y_pred = w0 + w1 * X_scaled
        error = y_pred - y_gd
        history.append(np.mean(error ** 2))
        
        dw0 = (2 / n) * np.sum(error)
        dw1 = (2 / n) * np.sum(error * X_scaled)
        
        w0 -= alpha * dw0
        w1 -= alpha * dw1
    
    return history

for alpha in [0.005, 0.05, 0.5]:
    history = run_gradient_descent(alpha)
    plt.plot(history, label=f"learning rate = {alpha}")

plt.xlabel("Epoch")
plt.ylabel("MSE Cost")
plt.title("Effect of Learning Rate")
plt.legend()
plt.grid(True)
plt.show()

## 14. Evaluating a Regression Model

After training, we must check how well the model performs.

Common regression metrics are:

| Metric | Meaning | Lower or Higher Better? |
|---|---|---|
| MAE | Mean Absolute Error | Lower is better |
| MSE | Mean Squared Error | Lower is better |
| RMSE | Root Mean Squared Error | Lower is better |
| R² Score | Amount of variation explained by the model | Higher is better |

### MAE

$$
MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|
$$

### MSE

$$
MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2
$$

### RMSE

$$
RMSE = \sqrt{MSE}
$$

### R² Score

R² tells us how much of the variation in the target value is explained by the model.

- R² close to 1 means strong fit.
- R² close to 0 means weak fit.
- R² can be negative if the model is worse than simply predicting the mean.

### In plain English

Evaluation metrics tell us whether the model is actually useful.

In [ ]:
y_pred_sklearn = model.predict(X)

mae = mean_absolute_error(y, y_pred_sklearn)
mse_value = mean_squared_error(y, y_pred_sklearn)
rmse = np.sqrt(mse_value)
r2 = r2_score(y, y_pred_sklearn)

pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Value": [mae, mse_value, rmse, r2]
})

## 15. Train-Test Split and Generalization

A model should not only perform well on training data.

It should also perform well on new unseen data.

This ability is called **generalization**.

To estimate generalization, we split data into:

| Dataset | Purpose |
|---|---|
| Training set | Used to learn model parameters |
| Test set | Used to evaluate performance on unseen data |

### In plain English

A good model should not just memorize old examples. It should predict well for new examples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42
)

model_split = LinearRegression()
model_split.fit(X_train, y_train)

y_train_pred = model_split.predict(X_train)
y_test_pred = model_split.predict(X_test)

print("Training R2:", round(r2_score(y_train, y_train_pred), 4))
print("Testing R2:", round(r2_score(y_test, y_test_pred), 4))
print("Testing RMSE:", round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 4))

## 16. Linear Basis Function Models

Sometimes the relationship between input and output is not a straight line.

For example, price may not increase linearly with mileage forever. There may be a curved relationship.

We can create new features from existing features, such as:

$$
x, x^2, x^3
$$

Then the model becomes:

$$
\hat{y} = w_0 + w_1x + w_2x^2 + w_3x^3
$$

This is called **polynomial regression**.

Important point:

> Polynomial regression is still a linear model because it is linear in the parameters $w_0, w_1, w_2, w_3$.

It is not linear in the original input $x$, but it is linear in the weights.

### In plain English

We can bend the prediction curve by creating polynomial features, but the model still learns weights linearly.

In [ ]:
# Create a non-linear looking dataset
np.random.seed(42)
X_curve = np.linspace(1, 10, 30).reshape(-1, 1)
y_curve = 2 + 0.7 * X_curve.ravel() + 0.18 * (X_curve.ravel() ** 2) + np.random.normal(0, 1.2, 30)

plt.scatter(X_curve, y_curve)
plt.xlabel("Feature x")
plt.ylabel("Target y")
plt.title("Data With a Curved Pattern")
plt.grid(True)
plt.show()

In [ ]:
# Fit simple linear regression
linear_model = LinearRegression()
linear_model.fit(X_curve, y_curve)
y_linear_pred = linear_model.predict(X_curve)

# Fit polynomial regression with degree 2
poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("linreg", LinearRegression())
])
poly_model.fit(X_curve, y_curve)
y_poly_pred = poly_model.predict(X_curve)

plt.scatter(X_curve, y_curve, label="Actual data")
plt.plot(X_curve, y_linear_pred, label="Linear regression")
plt.plot(X_curve, y_poly_pred, label="Polynomial regression degree 2")
plt.xlabel("Feature x")
plt.ylabel("Target y")
plt.title("Linear Model vs Polynomial Basis Model")
plt.legend()
plt.grid(True)
plt.show()

## 17. Underfitting and Overfitting

Two common problems in regression are **underfitting** and **overfitting**.

| Problem | Meaning | Example |
|---|---|---|
| Underfitting | Model is too simple | A straight line for strongly curved data |
| Overfitting | Model is too complex | A very high-degree polynomial that follows noise |

### Bias and variance intuition

| Concept | Meaning |
|---|---|
| High bias | Model is too simple and misses the true pattern |
| High variance | Model is too sensitive to training data noise |

### In plain English

Underfitting means the model learned too little. Overfitting means it learned the noise also.

In [ ]:
# Compare different polynomial degrees
for degree in [1, 2, 10]:
    pipeline = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("linreg", LinearRegression())
    ])
    pipeline.fit(X_curve, y_curve)
    y_pred = pipeline.predict(X_curve)
    plt.plot(X_curve, y_pred, label=f"degree {degree}")

plt.scatter(X_curve, y_curve, label="Actual data")
plt.xlabel("Feature x")
plt.ylabel("Target y")
plt.title("Model Complexity: Underfitting vs Overfitting")
plt.legend()
plt.grid(True)
plt.show()

## 18. Regularization

Regularization is a technique used to reduce overfitting.

It adds a penalty for large model weights.

The basic idea is:

> Keep the model accurate, but also keep the weights controlled.

Common regularized regression models are:

| Model | Penalty Type | Effect |
|---|---|---|
| Ridge Regression | L2 penalty | Shrinks weights toward small values |
| Lasso Regression | L1 penalty | Can make some weights exactly zero |
| Elastic Net | L1 + L2 penalty | Useful when features are correlated |

### Ridge Regression

Ridge adds a penalty based on squared weights.

$$
J(w) = MSE + \alpha \sum w_j^2
$$

### Lasso Regression

Lasso adds a penalty based on absolute weights.

$$
J(w) = MSE + \alpha \sum |w_j|
$$

### In plain English

Regularization tells the model: fit the data, but do not become unnecessarily complicated.

In [ ]:
# Regularization example with polynomial features
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)
}

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_curve, y_curve, test_size=0.3, random_state=42
)

results = []

for name, reg_model in models.items():
    pipeline = Pipeline([
        ("poly", PolynomialFeatures(degree=5, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", reg_model)
    ])
    pipeline.fit(X_train_c, y_train_c)
    pred = pipeline.predict(X_test_c)
    results.append({
        "Model": name,
        "Test_RMSE": np.sqrt(mean_squared_error(y_test_c, pred)),
        "Test_R2": r2_score(y_test_c, pred)
    })

pd.DataFrame(results)

## 19. Closed Form Solution vs Gradient Descent

Linear regression parameters can be learned in two major ways.

| Approach | Meaning | When Useful |
|---|---|---|
| Closed form solution | Direct mathematical formula | Smaller datasets, simple cases |
| Gradient descent | Iterative optimization | Large datasets, complex models |

### Closed form solution

The closed form solution computes the best parameters directly.

In matrix form:

$$
w = (X^TX)^{-1}X^Ty
$$

This is elegant, but it can become expensive when the number of features is very large.

### Gradient descent

Gradient descent updates parameters step by step. It is widely used in machine learning, especially when models are large.

### In plain English

Closed form is like solving the answer directly. Gradient descent is like improving the answer step by step.

## 20. Complete Mini Project: Car Price Prediction

Now I will put the main steps together:

1. Create data
2. Split into training and testing sets
3. Train a model
4. Predict
5. Evaluate
6. Interpret the model

In [ ]:
# Step 1: Create data
car_data = pd.DataFrame({
    "Mileage_kmpl": [10, 12, 14, 16, 18, 20, 22, 15, 13, 19, 21, 11],
    "Age_years":    [9, 8, 7, 6, 5, 4, 3, 6, 7, 4, 3, 9],
    "Distance_km":  [90000, 80000, 70000, 62000, 52000, 40000, 30000, 65000, 75000, 42000, 33000, 88000],
    "Price_lakhs":  [3.5, 4.2, 4.9, 5.8, 6.6, 7.6, 8.5, 5.4, 4.6, 7.2, 8.0, 3.8]
})

X = car_data[["Mileage_kmpl", "Age_years", "Distance_km"]]
y = car_data["Price_lakhs"]

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Step 3: Train model
final_model = LinearRegression()
final_model.fit(X_train, y_train)

# Step 4: Predict
predictions = final_model.predict(X_test)

# Step 5: Evaluate
print("MAE:", round(mean_absolute_error(y_test, predictions), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, predictions)), 4))
print("R2:", round(r2_score(y_test, predictions), 4))

# Step 6: Interpret parameters
pd.DataFrame({
    "Feature": ["Intercept"] + list(X.columns),
    "Parameter": [final_model.intercept_] + list(final_model.coef_)
})

In [ ]:
# Predict a new car price
new_car = pd.DataFrame({
    "Mileage_kmpl": [18],
    "Age_years": [5],
    "Distance_km": [50000]
})

predicted_price = final_model.predict(new_car)[0]
print("Predicted price:", round(predicted_price, 2), "lakhs")

## 21. Summary

In this chapter, I explained linear models for regression using a simple car price prediction example.

The most important points are:

| Concept | Meaning |
|---|---|
| Regression | Predicting a continuous numeric value |
| Linear model | A weighted sum of input features |
| Parameter | A value learned by the model |
| Cost function | A way to measure total prediction error |
| Least squares | Finds parameters that minimize squared error |
| Gradient descent | Learns parameters step by step |
| Learning rate | Controls step size in gradient descent |
| Evaluation metrics | Measure model quality |
| Polynomial regression | Uses transformed features but is still linear in weights |
| Regularization | Controls overfitting by penalizing large weights |

### In plain English

Linear regression is one of the simplest and most useful ML models. It learns a relationship between input features and a numeric output by finding the best-fitting line or surface.

## 22. Practice Exercises

Try these exercises after running the notebook.

### Exercise 1

Change the simple dataset and add more mileage-price records. Fit the model again. Check if the slope changes.

### Exercise 2

Predict the price for cars with mileage values of 11, 17, and 23 km/l.

### Exercise 3

In the multiple regression example, add a new feature called `Engine_CC`. Train the model again.

### Exercise 4

Try different learning rates in gradient descent:

- 0.001
- 0.01
- 0.1
- 1.0

Observe what happens to the cost curve.

### Exercise 5

Try polynomial regression with degrees 1, 2, 5, and 15. Observe underfitting and overfitting.

### Exercise 6

Try Ridge regression with different values of alpha:

- 0.001
- 0.01
- 0.1
- 1
- 10

Observe how the model performance changes.

## 23. Bridge to the Next Chapter

Linear regression predicts continuous values.

In the next chapter, we can move from regression to classification and study **linear models for classification**, especially **logistic regression**.

The key shift will be:

| Regression | Classification |
|---|---|
| Predicts a number | Predicts a class/category |
| Example: car price | Example: spam or not spam |
| Output is continuous | Output is discrete |